# NB2 - Formula-Based Drawdown-Penalised PPO

Reports the B2 alpha sweep. Each alpha is trained independently and evaluated through day-level Slurm arrays; `aggregate_final_results.py --select-b2` selects the primary B2 result.

## Snellius Pipeline Mode

This notebook is now a reporting/orchestration notebook. Heavy training and per-day evaluation are executed by Slurm scripts under `scripts/snellius/`; this notebook reads the resulting artifacts.

Expected artifacts:
- `results/b2_alpha0.1_test_results.csv`
- `results/b2_alpha1.0_test_results.csv`
- `results/b2_alpha10.0_test_results.csv`
- `results/b2_best_alpha.txt`
- `results/b2_test_results.csv`

Run from MobaXterm/Snellius with:

```bash
cd /home/<user>/thesis
export PROJECT_DIR=/home/<user>/thesis
export DATA_DIR=/scratch-shared/<user>/datasets
export CONDA_ENV=mysimenv
bash scripts/snellius/submit_final_pipeline.sh
```

In [ ]:
import json
import pathlib
import sys

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
META_DIR = RESULTS_DIR / "job_metadata"

pd.set_option("display.float_format", "{:.6f}".format)


def require_file(path, label=None, strict=False):
    path = pathlib.Path(path)
    if path.exists():
        print(f"OK      {label or path.name}: {path}")
        return True
    message = f"MISSING {label or path.name}: {path}"
    if strict:
        raise FileNotFoundError(message)
    print(message)
    return False


def read_csv(path, **kwargs):
    path = pathlib.Path(path)
    require_file(path, strict=True)
    return pd.read_csv(path, **kwargs)


def metric_summary(df):
    metrics = [c for c in ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "Mean |q|", "Near Cap Fraction"] if c in df.columns]
    return pd.DataFrame({"Mean": df[metrics].mean(), "Median": df[metrics].median(), "Std": df[metrics].std(ddof=1)})


def formula_metadata_files():
    if not META_DIR.exists():
        return []
    return sorted(META_DIR.glob("*.json"))

print(f"Project root : {PROJECT_ROOT}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Models dir   : {MODELS_DIR}")
print("Official Snellius execution path: bash scripts/snellius/submit_final_pipeline.sh")

## Artifact Preflight

In [ ]:
require_file(RESULTS_DIR / 'b2_alpha0.1_test_results.csv')
require_file(RESULTS_DIR / 'b2_alpha1.0_test_results.csv')
require_file(RESULTS_DIR / 'b2_alpha10.0_test_results.csv')
require_file(RESULTS_DIR / 'b2_best_alpha.txt')
require_file(RESULTS_DIR / 'b2_test_results.csv')

## Load Alpha Sweep Results

In [ ]:
alphas = [0.1, 1.0, 10.0]
frames = {}
for alpha in alphas:
    path = RESULTS_DIR / f"b2_alpha{alpha}_test_results.csv"
    if path.exists():
        frames[alpha] = read_csv(path)

summary_rows = []
for alpha, df in frames.items():
    row = {"alpha": alpha}
    row.update(metric_summary(df)["Mean"].to_dict())
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).set_index("alpha") if summary_rows else pd.DataFrame()
display(summary)

if (RESULTS_DIR / "b2_best_alpha.txt").exists():
    best_alpha = (RESULTS_DIR / "b2_best_alpha.txt").read_text().strip()
    print(f"Selected alpha: {best_alpha}")
    df_b2 = read_csv(RESULTS_DIR / "b2_test_results.csv")
    display(metric_summary(df_b2))

## Alpha Sweep Pareto View

In [ ]:
if not summary.empty:
    ax = summary.plot.scatter(x="Max DD", y="Sharpe", figsize=(7, 5))
    for alpha, row in summary.iterrows():
        ax.annotate(str(alpha), (row["Max DD"], row["Sharpe"]))
    ax.set_title("B2 alpha sweep: Sharpe vs Max DD")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "nb2_alpha_sweep_report.png", dpi=150, bbox_inches="tight")
    plt.show()